# KCSDB2 무역통계 DB — 주피터 노트북 시작

관세청 무역통계(2007.01–2026.03, 2,753만 거래행) 분석 환경.
이 노트북은 **DB 접속·구조 확인·쿼리 방법과 간단한 기술통계(descriptive)**를 시연하고,
**HS 개정을 건너 시계열을 잇는 방법**(5절)을 HS6·HS10 두 해상도로 보인다.
시연은 방법 예시이지 분석 방향이 아니다. 어떤 분석을 할지는 각자 자유롭게 정한다.

출처: 관세청·외교부 공공데이터(공공누리 제1유형 포함). 상세는 저장소 README 참조.

## 0. 사전 준비 (노트북 실행 전)

1. **패키지 설치** (터미널 또는 아래 셀):
   ```
   pip install duckdb pandas matplotlib jupyter
   ```
   재구축이 아니라 분석만 하므로 이 4개면 충분하다. conda·특정 파이썬 버전 불필요.

2. **DB 파일 배치**: 저장소 Releases에서 `kcsdb.duckdb.gz`를 받아 압축을 풀고
   `data/processed/kcsdb.duckdb`에 놓는다. (압축 해제: Windows는 7-Zip 등, Mac/Linux는 `gunzip kcsdb.duckdb.gz`)

3. 이 노트북을 저장소 루트에서 주피터로 연다: `jupyter notebook`

In [ ]:
# (선택) 패키지 설치 — 이미 설치했으면 건너뛴다
# !pip install duckdb pandas matplotlib

## 1. DB 파일 확인 및 접속

DB 경로를 확인한다. 파일이 없으면 위 0단계(Releases에서 받아 배치)를 먼저 한다.

In [ ]:
import os
import duckdb

# 저장소 안 어디서 노트북을 열어도 되도록 루트를 위로 올라가며 찾는다.
ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT:
        raise FileNotFoundError(
            "DB를 찾지 못했습니다: data/processed/kcsdb.duckdb\n"
            "Releases에서 kcsdb.duckdb.gz를 받아 압축 해제 후 data/processed/ 에 놓으세요."
        )
    ROOT = up

DB_PATH = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")
print("DB 경로:", DB_PATH)
print("DB 크기:", os.path.getsize(DB_PATH)//1024//1024, "MB")

# 읽기전용으로 연다. 원본을 건드리지 않고, 다른 노트북과 동시에 열 수 있다.
con = duckdb.connect(DB_PATH, read_only=True)
print("접속 완료")

## 2. 테이블·기간 확인

In [ ]:
print(con.sql("SHOW TABLES").df())
print("\nfact_trade 행수:", con.sql("SELECT COUNT(*) FROM fact_trade").fetchone()[0])
print("기간:", con.sql("SELECT MIN(yyyymm), MAX(yyyymm) FROM fact_trade").fetchone())

## 3. 스키마 확인

In [ ]:
for t in ["fact_trade", "fact_total", "dim_country", "dim_hs10",
          "dim_hs6_concordance", "dim_hs10_to_2022", "dim_hs10_name_hist"]:
    print(f"=== {t} ===")
    print(con.sql(f"DESCRIBE {t}").df().to_string(index=False))
    print()

## 4. 기술통계 시연 (descriptive)

아래는 **방법 시연**이다. 특정 가설·모형을 함축하지 않고 "데이터가 무엇을 담았는가"만 보인다.
**단위: 금액 미화 달러(USD), 중량 kg.**

주의(반드시 읽을 것):
- HS 시계열을 코드 동일성으로 이으면 2022 개정 경계에서 왜곡된다(dim_hs6_concordance 필요).
- 202603이 종점. 마지막 해/달을 추세로 읽지 말 것.
- 음수 중량 19행은 관세청 정정 원본. 단가 계산 시 이상치.

In [ ]:
import matplotlib.pyplot as plt

# (1) 연도별 총 수출·수입 (달러 USD) — 2026은 3월까지만이므로 제외
df1 = con.sql("""
    SELECT yyyymm // 100 AS year,
           SUM(exp_dlr) AS exports,
           SUM(imp_dlr) AS imports
    FROM fact_trade
    WHERE yyyymm // 100 < 2026
    GROUP BY 1 ORDER BY 1
""").df()
print(df1)

plt.figure(figsize=(9,4))
plt.plot(df1['year'], df1['exports']/1e9, marker='o', label='Exports')
plt.plot(df1['year'], df1['imports']/1e9, marker='s', label='Imports')
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade (2007-2025, excl. partial 2026)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# (2) 상위 교역국 (전 기간 누적 교역액) — dim_country 조인
# 표는 한글 국가명, 그림은 영문 국가명을 쓴다. 한글 폰트가 없는 환경에서도 그림이 깨지지 않게.
df2 = con.sql("""
    SELECT d.name_ko_kcs AS country,
           d.name_en     AS country_en,
           SUM(f.exp_dlr + f.imp_dlr) AS trade
    FROM fact_trade f
    LEFT JOIN dim_country d ON f.stat_cd = d.stat_cd
    GROUP BY 1, 2 ORDER BY trade DESC LIMIT 10
""").df()
print(df2[['country', 'trade']])

plt.figure(figsize=(9,4))
plt.barh(df2['country_en'][::-1], df2['trade'][::-1]/1e9)
plt.xlabel('Billion USD (cumulative 2007-2026.03)')
plt.title('Top 10 Trade Partners')
plt.tight_layout(); plt.show()

In [ ]:
# (3) HS2 대분류별 수출 상위 10 — hs10을 hs2로 절단(dim 없이 SUBSTR)
df3 = con.sql("""
    SELECT SUBSTR(hs10,1,2) AS hs2,
           SUM(exp_dlr) AS exports
    FROM fact_trade
    GROUP BY 1 ORDER BY exports DESC LIMIT 10
""").df()
print(df3)

plt.figure(figsize=(9,4))
plt.bar(df3['hs2'], df3['exports']/1e9)
plt.ylabel('Billion USD'); plt.xlabel('HS2 code')
plt.title('Top 10 HS2 Categories by Export (cumulative)')
plt.tight_layout(); plt.show()
print("\nHS2 코드의 품목명은 dim_hs10이 hs10 단위라 직접 없음.")
print("필요시 HS 분류표 참조(예: 85=전기기기, 87=차량). 분석 목적에 따라 해석.")

In [ ]:
# (4) 연도별 무역수지 (수출-수입)
df4 = con.sql("""
    SELECT yyyymm // 100 AS year,
           SUM(exp_dlr - imp_dlr) AS balance
    FROM fact_trade
    WHERE yyyymm // 100 < 2026
    GROUP BY 1 ORDER BY 1
""").df()
print(df4)

plt.figure(figsize=(9,4))
colors = ['crimson' if b < 0 else 'steelblue' for b in df4['balance']]
plt.bar(df4['year'], df4['balance']/1e9, color=colors)
plt.axhline(0, color='black', lw=0.8)
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade Balance (2007-2025)')
plt.tight_layout(); plt.show()

## 5. HS 개정 연결 (필수)

HS 코드는 2012·2017·2022년 1월에 개정됐다. 개정 때 코드가 생기고 없어지고 쪼개지고 합쳐지므로,
**코드가 같다는 것만으로 시계열을 이으면 개정 경계에서 계열이 끊긴다.** 끊긴 자리는 품목이
사라진 것처럼 보이지만 실제로는 번호만 바뀐 것이다.

이 DB는 연계표를 두 해상도로 담고 있다. HS6는 관세청이 공식 공표한 것이고,
HS10은 이 프로젝트가 별표(품목분류표)를 대조해 추정한 것이다. 아래에서 차례로 쓴다.

### 5.1 HS6 — 관세청 공식 연계표 (dim_hs6_concordance)

`dim_hs6_concordance`는 과거 판본의 HS6를 2022년 기준으로 잇는다. `relation='mapped'`가
실제로 대응이 있는 행이다. 한 코드가 여러 코드로 갈리는 다대다 관계가 있으므로,
집계할 때 중복이 생기지 않는지 확인해야 한다. 아래는 판본별로 몇 개 코드가 연결되는지 센 것이다.

In [ ]:
# 과거 hs6를 hs2022 기준으로 환산. 다대다이므로 실제 분석 시 중복 처리 필요.
df5 = con.sql("""
    SELECT past_version,
           COUNT(DISTINCT hs_past) AS past_codes,
           COUNT(DISTINCT hs2022) AS mapped_2022
    FROM dim_hs6_concordance
    WHERE relation = 'mapped'
    GROUP BY 1 ORDER BY 1
""").df()
print(df5)

### 5.2 HS10 — 추정 연계표 (dim_hs10_to_2022)

HS6는 믿을 수 있지만 6자리로 뭉뚱그려진다. 품목을 10자리로 봐야 하는 분석이라면
`dim_hs10_to_2022`를 쓴다. 다만 성격이 다르다는 것을 알고 써야 한다.

**HS10 승계표는 어디에도 공표되지 않는다.** 관세청도 기재부도 "무엇이 무엇을 이어받았다"를
발표하지 않는다. 이 표는 연도별 품목분류표를 대조해 코드의 생몰을 알아내고, 개정 전후 수출입액이
맞아떨어지도록 가중치를 추정한 것이다. 따라서 **HS6 연계표와 달리 추정치이며 오차가 있다.**

쓰는 법은 세 가지만 지키면 된다.

1. **`method='chain'`으로 거른다.** `hs6_fallback`은 10자리 경로를 찾지 못해 6자리로 메운 보완분이라
   섞으면 HS10 해상도가 무너진다.
2. **과거 시점의 판본(`past_version`)을 맞춘다.** 2011년까지는 `'2007'`, 2012~2016년은 `'2012'`,
   2017~2021년은 `'2017'`이고, 2022년 이후 자료는 이미 현행이라 변환하지 않는다.
3. **금액에 `weight`를 곱해 다시 집계한다.** 한 과거 코드의 가중치 합은 1이므로 총액은 보존된다.

먼저 표가 어떻게 생겼는지 본다.

In [ ]:
print(con.sql("""
    SELECT past_version, method,
           COUNT(*) AS rows,
           COUNT(DISTINCT hs_past) AS past_codes,
           COUNT(DISTINCT hs2022)  AS to_2022
    FROM dim_hs10_to_2022
    GROUP BY 1, 2 ORDER BY 1, 2
""").df().to_string(index=False))

# 한 과거 코드의 가중치 합은 1이다(chain 기준). 총액이 보존된다는 뜻.
print("\n가중치 합 점검(2017판 chain):")
print(con.sql("""
    SELECT ROUND(MIN(s), 6) AS 최소, ROUND(MAX(s), 6) AS 최대, COUNT(*) AS 코드수
    FROM (SELECT hs_past, SUM(weight) AS s FROM dim_hs10_to_2022
          WHERE past_version = '2017' AND method = 'chain' GROUP BY 1)
""").df().to_string(index=False))

연계표가 실제로 무엇을 고치는지 보려면 개정을 사이에 둔 두 해를 비교하면 된다.
아래는 2019년과 2023년 수출을 HS10 단위로 맞춰 보면서, 코드를 그대로 쓸 때와
연계표를 적용할 때 **양쪽 해에 모두 존재하는 코드 수**와 **2019년에만 있고 2023년에는 사라진 금액**이
어떻게 달라지는지를 잰 것이다. 사라진 금액은 776억 달러에서 0.4억 달러로 줄어든다.
2022년 개정으로 번호가 바뀐 것을 소멸로 오해하고 있었다는 뜻이다.

In [ ]:
print(con.sql("""
WITH y19 AS (SELECT hs10, SUM(exp_dlr) AS v FROM fact_trade
             WHERE yyyymm BETWEEN 201901 AND 201912 GROUP BY 1),
     y23 AS (SELECT hs10, SUM(exp_dlr) AS v FROM fact_trade
             WHERE yyyymm BETWEEN 202301 AND 202312 GROUP BY 1),
     conv AS (SELECT c.hs2022 AS hs10, SUM(y.v * c.weight) AS v
              FROM y19 y
              JOIN dim_hs10_to_2022 c
                ON c.hs_past = y.hs10 AND c.past_version = '2017' AND c.method = 'chain'
              GROUP BY 1)
SELECT '코드 그대로' AS 방식,
       COUNT(*) FILTER (WHERE a.v IS NOT NULL AND b.v IS NOT NULL) AS 양쪽존재,
       ROUND(SUM(a.v) FILTER (WHERE b.v IS NULL) / 1e8, 1) AS "2019년에만_억달러"
FROM y19 a FULL JOIN y23 b USING (hs10)
UNION ALL
SELECT '연계표 적용',
       COUNT(*) FILTER (WHERE a.v IS NOT NULL AND b.v IS NOT NULL),
       ROUND(SUM(a.v) FILTER (WHERE b.v IS NULL) / 1e8, 1)
FROM conv a FULL JOIN y23 b USING (hs10)
""").df().to_string(index=False))

전 기간을 한 번에 2022년 기준으로 옮기려면 아래 패턴을 쓴다. 요령은 **먼저 집계하고 나중에
조인하는 것**이다. 2,753만 행에 곧바로 연계표를 붙이면 몇 분이 걸리지만, 필요한 단위로
줄인 뒤 붙이면 몇 초에 끝난다. 여기서는 (연도, HS10)으로 줄였다.
2022년 이후 행은 이미 현행 코드이므로 `LEFT JOIN`이 비고, `COALESCE`가 원래 코드와 가중치 1을 넣는다.

In [ ]:
import time
t0 = time.time()
hs2022_year = con.sql("""
WITH agg AS (                       -- 먼저 줄인다: 2,753만행 -> 수십만행
  SELECT yyyymm // 100 AS year, hs10, SUM(exp_dlr) AS v,
         CASE WHEN yyyymm < 201201 THEN '2007'
              WHEN yyyymm < 201701 THEN '2012'
              WHEN yyyymm < 202201 THEN '2017' END AS pv
  FROM fact_trade WHERE exp_dlr > 0
  GROUP BY 1, 2, 4
)
SELECT a.year,
       COALESCE(c.hs2022, a.hs10)     AS hs2022,     -- 2022년 이후는 그대로
       SUM(a.v * COALESCE(c.weight, 1)) AS exports
FROM agg a
LEFT JOIN dim_hs10_to_2022 c
  ON c.hs_past = a.hs10 AND c.past_version = a.pv AND c.method = 'chain'
GROUP BY 1, 2
""").df()
print(f"{len(hs2022_year):,}행, {time.time()-t0:.1f}초")

# 총액은 보존되어야 한다(가중치 합이 1이므로).
chk = con.sql("SELECT yyyymm//100 AS year, SUM(exp_dlr) AS raw FROM fact_trade "
              "WHERE exp_dlr > 0 GROUP BY 1").df()
m = hs2022_year.groupby('year', as_index=False)['exports'].sum().merge(chk, on='year')
print("총액 오차 최대(%):", round(((m.exports/m.raw - 1)*100).abs().max(), 6))
print("\n연도별 코드 수(2022년 기준으로 통일된 뒤):")
print(hs2022_year.groupby('year').hs2022.nunique().tail(6).to_string())

**금액이 아니라 개수를 세는 지표라면 안분을 쓰면 안 된다.** 목적지 수, 품목 수, 진입·퇴출처럼
"몇 개인가"를 재는 지표에서 한 과거 코드를 여러 승계자에게 쪼개 넣으면, 그 코드가 가졌던
거래 상대국 집합이 승계자 전부에게 복제되어 개수가 부풀고 개정월에 가짜 단차가 생긴다.

이때는 **가중치가 가장 큰 승계자 하나에 통째로 배정하는 방식**(최빈 승계)을 쓴다. 대신 금액이
비례 몫대로 실리지 않는 것을 감수한다. 아래는 최빈 승계표를 만들고, 얼마나 깨끗하게 갈리는지
(가중치 중위값과 애매한 계열 수) 확인하는 코드다.

In [ ]:
dominant = con.sql("""
    SELECT hs_past, past_version, hs2022, weight FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY hs_past, past_version
                                   ORDER BY weight DESC, hs2022) AS rn
      FROM dim_hs10_to_2022 WHERE method = 'chain'
    ) WHERE rn = 1
""").df()
print(f"최빈 승계표 {len(dominant):,}행")
print(dominant.groupby('past_version').weight
      .agg(중위='median', 평균='mean',
           애매=lambda s: int((s < 0.9).sum())).round(3).to_string())
print("\n'애매'는 최빈 승계자의 몫이 90% 미만인 계열이다. 이 계열은 해석에 주의한다.")


마지막으로 폐지된 코드의 품목명이다. `dim_hs10`은 현행 코드만 담고 있어 과거에만 존재했던
코드를 조회하면 이름이 나오지 않는다. 품목명이 판본마다 바뀌기 때문에 한 코드에 이름을 하나만
붙일 수 없어서, 이력을 따로 `dim_hs10_name_hist`(코드 x 별표 연도)에 담았다.

아래는 지금은 없어졌지만 과거 거래액이 컸던 코드 열 개의 이름을 되찾은 것이다.
품목명이 "기타"로 나오는 경우가 많은데 이는 자료의 결함이 아니다. HS 품명은 상위 분류 아래에서
상대적으로 붙기 때문에, 같은 호(號)에 열거된 품목 다음에 오는 나머지를 "기타"로 적는다.
따라서 폐지코드의 이름은 상위 6자리 분류와 함께 읽어야 뜻이 통한다.

In [ ]:
print(con.sql("""
WITH dead AS (                                   -- 현행에 없는 = 폐지된 코드
    SELECT hs10, SUM(exp_dlr + imp_dlr) AS v
    FROM fact_trade
    WHERE hs10 NOT IN (SELECT hs10 FROM dim_hs10)
    GROUP BY 1 ORDER BY v DESC LIMIT 10
)
SELECT d.hs10,
       SUBSTR(d.hs10, 1, 6)        AS hs6,
       ROUND(d.v / 1e8, 0)         AS 누적교역_억달러,
       MIN(h.byeolpyo_year)        AS 최초별표,
       MAX(h.byeolpyo_year)        AS 최종별표,
       ANY_VALUE(h.name_ko)        AS 품목명
FROM dead d
LEFT JOIN dim_hs10_name_hist h ON h.hs10 = d.hs10
GROUP BY 1, 2, 3 ORDER BY 누적교역_억달러 DESC
""").df().to_string(index=False))

print("")
print("한 코드의 이름이 별표 연도마다 다를 수 있다. 시점에 맞는 연도를 골라 쓴다.")

## 6. 분석 시 주의 (필독)

- **대용량 결과를 .df()로 통째 가져오지 말 것.** 집계를 SQL 안에서 끝내고 소형 결과만 가져온다.
  연계표를 붙일 때도 마찬가지다. 먼저 필요한 단위로 집계하고 그다음에 조인한다(5.2절).
- **HS 개정 연결**: 코드 동일성만으로 시계열을 이으면 개정 경계에서 계열이 끊긴다.
  HS6면 `dim_hs6_concordance`(공식), HS10이면 `dim_hs10_to_2022`(추정, `method='chain'`)를 쓴다.
- **HS10 연계표는 추정치다.** 공식 승계표가 존재하지 않아 별표 대조로 만들었다. 금액 기준
  분석에는 충분하지만, 개별 코드 하나의 승계 경로를 근거로 삼는 주장은 피한다.
- **개수를 세는 지표에는 안분을 쓰지 말 것.** 목적지 수·품목 수·진입·퇴출처럼 개수를 재는
  지표는 최빈 승계로 바꿔야 한다(5.2절). 안분하면 개수가 부풀고 개정월에 가짜 단차가 생긴다.
- **폐지된 코드의 품목명**은 `dim_hs10`이 아니라 `dim_hs10_name_hist`에 있다.
- **음수 중량 19행**: 관세청 정정 원본. 단가 계산 시 이상치.
- **범위 경계**: 202603까지. 2026년은 부분년(1–3월)이므로 연도 비교 시 제외했다.
- **단가**는 월별 단가의 평균이 아니라 금액 합을 중량 합으로 나눠 구한다.
  수출은 FOB, 수입은 CIF라 수준을 맞대 비교할 수 없고 변화율만 비교한다.
- 상세 함정은 저장소 `docs/세션_발견_노트.md`와 대시보드의 "HS 연계" 탭 참조.

## 7. 마무리

분석이 끝나면 연결을 닫는다.

In [ ]:
con.close()